# DocGuard — Phase 4 Transformer comparison

This notebook fine-tunes a pretrained MiT-B0 encoder with a custom segmentation decoder and trains a matching-resolution CNN baseline. Synthetic-only data; results are unknown until executed. Select a GPU runtime. Retain the upstream research/evaluation license with derived weights.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
assert "docguard_colab.zip" in uploaded, "Upload docguard_colab.zip"
work = Path("/content/docguard")
work.mkdir(exist_ok=True)
with zipfile.ZipFile("docguard_colab.zip") as archive:
    for name in archive.namelist():
        assert (work/name).resolve().is_relative_to(work.resolve()), "Unsafe archive path"
    archive.extractall(work)
os.chdir(work)
print("Project extracted:", work)

In [ ]:
%pip install -q -r requirements-transformer.txt
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before training"
print(torch.cuda.get_device_name(0))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/validate_dataset.py", "data/synthetic_v2"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

## Train both models before viewing test results
Both use 320×400 images and 20 epochs, but pretraining, learning rates and batch sizes differ. This is a practical baseline comparison, not an architecture-only experiment. Encoder fine-tuning is enabled for the Transformer.

In [ ]:
CNN = "runs/cnn_phase4_colab"
TRANSFORMER = "runs/transformer_phase4_colab"
subprocess.run([sys.executable, "-m", "docguard.baseline", "train", "--output", CNN, "--epochs", "20"], check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "docguard.baseline", "train", "--model", "transformer", "--output", TRANSFORMER, "--epochs", "20", "--batch-size", "4", "--lr", "0.00006"], check=True)

In [ ]:
for run in [CNN, TRANSFORMER]:
    subprocess.run([sys.executable, "-m", "docguard.baseline", "test", "--output", run, "--batch-size", "4"], check=True)
subprocess.run([sys.executable, "scripts/compare_models.py", CNN, TRANSFORMER], check=True)

In [ ]:
from IPython.display import display, Image
for path in sorted(Path(TRANSFORMER, "predictions").glob("*.png"))[:4]:
    display(Image(filename=str(path)))

In [ ]:
import shutil
export = Path("/content/docguard_phase4_export")
export.mkdir(exist_ok=True)
for run in [CNN, TRANSFORMER]:
    shutil.copytree(run, export/Path(run).name, dirs_exist_ok=True)
shutil.copy("deliverables/model_comparison.json", export)
shutil.copy("docs/third_party/SEGFORMER_LICENSE.txt", export)
archive = shutil.make_archive("/content/docguard_phase4_results", "zip", export)
files.download(archive)

## Review gate
Inspect global F1/IoU, original-document false alarms, subgroup metrics and prediction examples. Do not claim improvement unless measured. This notebook has not been executed in your Colab account. Clean synthetic edits may lack visual evidence; results do not establish real-document reliability.